# 06 - Feature Store Retrieval & REST Inference

End-to-end check of the **deployed PROD model** from a client's point of view:

1. Fetch features for a set of entities from the **Feature Store** (`retrieve_feature_values`)
2. Look up the **Gateway** (stable URL) and the service behind it
3. Call the model over **HTTPS REST** (`dataframe_split`, PAT auth)
4. Check the REST predictions match in-Snowflake scoring (`ModelVersion.run`)
5. Measure latency

**Prerequisites**
- The deploy workflow has run through `deploy-prod` with `serving.realtime.enabled: true` (a service and gateway exist in PROD).
- A **programmatic access token (PAT)** for a user whose role can call the endpoint (the service owner `MLOPS_PROD_ROLE`, or a role granted the service role `ALL_ENDPOINTS_USAGE`). Export it before starting Jupyter:
  ```bash
  export SNOWFLAKE_PAT=<token_secret>
  ```
  Create one with `ALTER USER ADD PROGRAMMATIC ACCESS TOKEN <name> ROLE_RESTRICTION = 'MLOPS_PROD_ROLE' DAYS_TO_EXPIRY = 1;`. If the variable isn't set, the notebook prompts for it. Never paste a PAT into a cell.

> The ingress rejects connector session tokens for this endpoint (it returns HTTP 500 rather than 401), so a PAT is required. Authorization failures return **404**.

In [ ]:
import sys
sys.path.insert(0, "../source")

import getpass
import json
import os
import time

import numpy as np
import pandas as pd
import requests

from config import (
    FEATURE_VIEW_NAME,
    FEATURE_VIEW_VERSION,
    ID_COLUMNS,
    MODEL_NAME,
    PREDICT_FUNCTION,
    REALTIME_CONFIG,
    SCHEMA,
    SOURCE_DATABASE,
    SOURCE_SCHEMA,
    TASK_TYPE,
    env_setting,
)
from snowpark_session import create_snowpark_session

# This notebook targets PROD: the deployed model, its feature views, and gateway.
PROD_DATABASE = env_setting("prod", "database")
PROD_WAREHOUSE = env_setting("prod", "warehouse")
GATEWAY_NAME = REALTIME_CONFIG["gateway_name"]
N_ENTITIES = 10  # entities to score

session = create_snowpark_session()
session.use_warehouse(PROD_WAREHOUSE)
print(f"Connected as {session.get_current_user()} ({session.get_current_role()})")
print(f"Model: {PROD_DATABASE}.{SCHEMA}.{MODEL_NAME} | features: {FEATURE_VIEW_NAME}/{FEATURE_VIEW_VERSION}")

## Step 1: Fetch features from the Feature Store

Build a **spine** (the entity keys to score) and let the Feature Store join the latest feature values for each entity. This is the same feature view version the PROD model was trained on (`feature_store.view_version` in `config.yml`), so the inputs match the model's training features.

In [ ]:
from snowflake.ml.feature_store import CreationMode, FeatureStore

fs = FeatureStore(
    session=session,
    database=PROD_DATABASE,
    name=SCHEMA,
    default_warehouse=PROD_WAREHOUSE,
    creation_mode=CreationMode.FAIL_IF_NOT_EXIST,
)
fv = fs.get_feature_view(FEATURE_VIEW_NAME, FEATURE_VIEW_VERSION)
print(f"Feature view: {fv.name}/{fv.version} | status={fv.status} | refresh={fv.refresh_freq}")

# Spine: the entity keys we want predictions for
spine = (
    session.table(f"{SOURCE_DATABASE}.{SOURCE_SCHEMA}.CUSTOMER_PROFILES")
    .select(*ID_COLUMNS)
    .limit(N_ENTITIES)
)

features_df = fs.retrieve_feature_values(spine_df=spine, features=[fv]).to_pandas()
print(f"Retrieved {len(features_df)} rows x {features_df.shape[1]} columns")
features_df.head()

## Step 2: Shape the request from the model signature

The registered model's signature defines the exact input columns and types the endpoint expects. Select those columns from the retrieved features (dropping keys and timestamps) and cast to the signature types, the same way the pipeline's batch scoring and health checks do.

In [ ]:
from snowflake.ml.registry import Registry
from serving.signature import cast_pandas, input_features, output_columns

model = Registry(session=session, database_name=PROD_DATABASE, schema_name=SCHEMA).get_model(MODEL_NAME)
mv = model.default
print(f"Default PROD version: {mv.version_name}")
print(f"  comment (lineage): {mv.comment}")

feature_cols = input_features(mv, PREDICT_FUNCTION)
missing = [c for c in feature_cols if c not in features_df.columns]
assert not missing, f"Feature view is missing model inputs: {missing}"

X = cast_pandas(features_df[feature_cols].copy(), mv, PREDICT_FUNCTION)
print(f"\nModel inputs ({len(feature_cols)}): {feature_cols}")
print(f"Model outputs: {output_columns(mv, PREDICT_FUNCTION)}")
X.dtypes

## Step 3: Find the endpoint

Clients should call the **Gateway**, the stable URL that stays the same across model versions while CI shifts traffic between per-version services (`gateway-config.yml`). The service's own URL is shown too, for debugging a specific version.

A newly created gateway reports `Endpoints provisioning in progress...` for a few minutes; rerun the cell until it shows a hostname.

In [ ]:
gw = session.sql(f"DESC GATEWAY {PROD_DATABASE}.{SCHEMA}.{GATEWAY_NAME}").collect()[0]
gateway_host = gw["ingress_url"]
assert gateway_host and "provisioning" not in gateway_host.lower(), f"Gateway not ready yet: {gateway_host}"

# Services (and weights) behind the gateway
targets = []
for line in gw["spec"].splitlines():
    line = line.strip()
    if line.startswith("value:"):
        targets.append({"service": line.split("value:")[1].split("!")[0].strip().split(".")[-1]})
    elif line.startswith("weight:") and targets:
        targets[-1]["weight"] = int(line.split("weight:")[1])
print(f"Gateway {GATEWAY_NAME}: https://{gateway_host}")
for t in targets:
    print(f"  -> {t['service']}: {t.get('weight')}%")

# Direct URL of the service for the default version (public 'inference' endpoint)
service_name = f"{MODEL_NAME}_SERVICE_{mv.version_name}"
endpoints = session.sql(f"SHOW ENDPOINTS IN SERVICE {PROD_DATABASE}.{SCHEMA}.{service_name}").collect()
service_host = next(r["ingress_url"] for r in endpoints if r["name"] == "inference")
print(f"Service {service_name}: https://{service_host}")

# Method names map to URL paths with underscores as dashes: predict_proba -> /predict-proba
METHOD_PATH = "/" + PREDICT_FUNCTION.replace("_", "-")
GATEWAY_URL = f"https://{gateway_host}{METHOD_PATH}"
SERVICE_URL = f"https://{service_host}{METHOD_PATH}"

## Step 4: Call the model over REST

Build the `dataframe_split` body with pandas (`to_json(orient="split")`) so types serialize consistently, then POST it with the PAT in the `Authorization: Snowflake Token="..."` header.

The response is `{"data": [[row_index, {output_column: value, ...}], ...]}`, one entry per input row in the same order.

In [ ]:
PAT = os.getenv("SNOWFLAKE_PAT") or getpass.getpass("Snowflake PAT: ")
HEADERS = {"Authorization": f'Snowflake Token="{PAT}"', "Content-Type": "application/json"}


def predict_rest(url: str, X: pd.DataFrame, timeout: int = 60) -> pd.DataFrame:
    """POST a DataFrame to a model endpoint and return outputs as a DataFrame."""
    payload = {"dataframe_split": json.loads(X.to_json(orient="split"))}
    resp = requests.post(url, headers=HEADERS, json=payload, timeout=timeout)
    if resp.status_code == 404:
        raise RuntimeError("404: wrong URL, or the PAT's role lacks endpoint access (grant ALL_ENDPOINTS_USAGE)")
    resp.raise_for_status()
    rows = sorted(resp.json()["data"], key=lambda r: r[0])
    return pd.DataFrame([r[1] for r in rows], index=X.index)


start = time.perf_counter()
rest_preds = predict_rest(GATEWAY_URL, X)
print(f"Gateway call: {len(X)} rows in {(time.perf_counter() - start) * 1000:.0f} ms")

result = pd.concat([features_df[ID_COLUMNS], rest_preds], axis=1)
result

## Step 5: Validate the predictions

- **Output validity**: the same checks CI runs (for classifiers, probabilities in [0, 1] that sum to 1).
- **Parity**: REST outputs should match scoring the same rows in a warehouse with `ModelVersion.run`. Small float differences (around 1e-6) are expected; the deploy health check allows up to 1e-4.

If the gateway splits traffic across versions (a canary), rows can be served by different versions, so the parity check goes to the default version's service directly.

In [ ]:
from serving.signature import validate_outputs

out_cols = output_columns(mv, PREDICT_FUNCTION)

problems = validate_outputs(TASK_TYPE, rest_preds[out_cols].to_numpy())
print("Output validity:", "OK" if not problems else problems)

# Parity: default-version service over REST vs. warehouse scoring of the same rows
service_preds = predict_rest(SERVICE_URL, X)
warehouse_preds = mv.run(X, function_name=PREDICT_FUNCTION)[out_cols]
max_diff = float(np.max(np.abs(service_preds[out_cols].to_numpy() - warehouse_preds.to_numpy())))
print(f"Max |REST - warehouse| = {max_diff:.2e}")
assert max_diff <= 1e-4, "REST and warehouse predictions disagree"
print("Parity: OK")

## Step 6: Latency

Single-row requests approximate an online use case such as scoring one transaction. The first call can be slower (connection setup). Latency here includes the network path from this machine to Snowflake.

In [ ]:
latencies_ms = []
for i in range(len(X)):
    start = time.perf_counter()
    predict_rest(GATEWAY_URL, X.iloc[[i]])
    latencies_ms.append((time.perf_counter() - start) * 1000)

lat = pd.Series(latencies_ms)
print(f"Single-row gateway latency over {len(lat)} calls: "
      f"p50={lat.median():.0f} ms | p95={lat.quantile(0.95):.0f} ms | max={lat.max():.0f} ms")

In [ ]:
session.close()
print("Notebook complete.")